## Runnable Class, RunnableSequenceClass, RunnablePassThroughClass 
### PIPING CHAINS

In [13]:
import config
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [14]:
RunnablePassthrough().invoke('HI')

'HI'

In [15]:
RunnablePassthrough().invoke([1,2,3])

[1, 2, 3]

## our main task.
### Piping chains together.
### In the following example, we'll create a chain that lists the most essential tools for a given profession.
### Then we'll feed this output to a second chain, whose task will be to suggest effective strategies for
### mastering these tools.

In [16]:
chat_template_tools = ChatPromptTemplate.from_template(''' 
What are the five most important tools a {job title} needs?
Answer only by listing the tools here.
''')
chat_template_tools #Notice that by default, this new method assumes that the template we passed as an argument is a human message prompt template. This will ease our implementation even more.

ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template=' \nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools here.\n'), additional_kwargs={})])

In [17]:
chat_template_strategy = ChatPromptTemplate.from_template(''' 
Considering the tools provided, develop a strategy for effectively learning and mastering them: {tools}
''')

In [18]:
chat = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    seed = 365,
    max_tokens = 30,
    openai_api_key = config.api_key
)

In [20]:
string_parser =  StrOutputParser()

In [27]:
chain_tools = chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()}
chain_strategy = chat_template_strategy | chat | string_parser

In [22]:
print(chain_tools.invoke({'job title': 'data scientist'}))

1. Programming languages (e.g. Python, R)
2. Statistical software (e.g. SAS, SPSS)
3. Data visualization tools


In [24]:
print(chain_strategy.invoke({'tools': ''' 
1. Programming languages (e.g. Python, R)
2. Statistical software (e.g. SAS, SPSS)
3. Data visualization tools
'''}))

1. Programming languages (e.g. Python, R):
- Start by understanding the basics of programming such as variables, data types, loops, and


###  now how can we merge these two chains into one?
Note that the first component of chain strategy accepts a dictionary with tools being the key and the
value being the output from the first chain.
Therefore, to connect the two chains, we must create this dictionary at the end of the first chain
to input the second one.
This is where the runnable passthrough class comes into play.

In [28]:
#chain_tools = chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()}
#The resulting string serves as an input for the runnable passthrough.

#The final component mirrors its input, thereby outputting a dictionary that maps tools to the same string.

In [29]:
print(chain_tools.invoke({'job title': 'doctor'}))

{'tools': '1. Stethoscope\n2. Blood pressure cuff\n3. Thermometer\n4. Otoscope\n5. Reflex hammer'}


In [30]:
chain_combined = chain_tools | chain_strategy

In [31]:
print(chain_combined.invoke({'job title': 'AIML Intern'}))

1. Programming languages:
- Start by choosing one programming language to focus on initially. Python is often recommended for beginners due to its readability and versatility.



you can see one of the significant advantages of the long chain expression language its modularity,
which allows us to create these bite sized chains and then combine them to form longer ones.

OR 
chain_combined = (chain_tools = chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()} |
                    chain_strategy = chat_template_strategy | chat | string_parser
)

Now, this short but valuable lesson focuses on visualizing these chains.
We'll utilize a Python library (grandalf) that allows us to draw and understand the processes inside a chain even
better.